In [8]:
import torch
from torch_scatter import scatter_mean
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import networkx as nx
import os
import pickle
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

from methods import mcmc_community_delays, mmca_community_delays
from methods.utils import graph_dynamic_delays

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [9]:
# ---------------初始数据-----------------------------

# 导入默认参数
file_path = os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), 'parameters/community_data_1.pkl')
with open(file_path, 'rb') as f:
    init_data = pickle.load(f)

epi_paras = torch.tensor(init_data['epi_paras'], dtype=torch.float32).to(device)
soc_paras = torch.tensor(init_data['soc_paras'], dtype=torch.float32).to(device)
features_state = torch.tensor(init_data['init_state'], dtype=torch.float32).to(device)
P_rows, P_cols = init_data['P_matrix'].nonzero()
P_edge_index = torch.tensor(np.array([P_rows, P_cols]), dtype=torch.long).to(device)
communities = init_data['communities']
P_community = init_data['P_community']
P_G = init_data['P_G']

I_rows, I_cols = init_data['I_matrix'].nonzero()
I_edge_index = torch.tensor(np.array([I_rows, I_cols]), dtype=torch.long).to(device)

# # # ---------------初始数据-----------------------------
node_num =  features_state.shape[0]
time_scale = 400

In [10]:
Tscale = 40
# mcmc_soc_file = f'./data/mcmc_soc_attention_10_10_5.pt'
# mcmc_soc_file = f'./data/mcmc_soc_attention_10_X_5.pt'
mcmc_soc_file = f'./data/mcmc_soc_attention_10_10_X.pt'
# mcmc_soc_file = f'./data/mcmc_soc_attention_10_X_X.pt'

mcmc_soc_attention = torch.load(mcmc_soc_file)
mcmc_soc_attention = torch.mean(mcmc_soc_attention[0:Tscale*10+1,...], dim=-1)
mcmc_soc_attention_con = torch.mean(mcmc_soc_attention, dim=0).unsqueeze(0).repeat(mcmc_soc_attention.shape[0],1,1)


C:\Users\95406\AppData\Local\Temp\ipykernel_42784\3905414934.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mcmc_soc_attention = torch.load(mcmc_soc_file)


In [11]:
para_len = 100
soc_paras_mcmc = soc_paras.unsqueeze(0).repeat(para_len, 1)
epi_paras_mcmc = epi_paras.unsqueeze(0).repeat(para_len, 1, 1) 
features_state_tensor_mcmc = features_state.unsqueeze(0).repeat(para_len, 1, 1).to(device)
delays = 0
# sigmoid = None  # sigmoid参数
sigmoid = [10,10,5]  # sigmoid参数
mcmc = mcmc_community_delays.MCMC(para_len, device)
mcmc_features_times, _, mcmc_soc_attention = graph_dynamic_delays(time_scale, mcmc, features_state_tensor_mcmc.clone(), epi_paras_mcmc,soc_paras_mcmc, P_edge_index, I_edge_index, communities, device, delays,  sigmoid = sigmoid, soc_attention_constant = mcmc_soc_attention_con)
# torch.save(mcmc_features_times, f'./data/mcmc_features_times_con_{sigmoid[0]}_{sigmoid[1]}_{sigmoid[2]}.pt')
# torch.save(mcmc_soc_attention, f'./data/mcmc_soc_attention_con_{sigmoid[0]}_{sigmoid[1]}_{sigmoid[2]}.pt')

# torch.save(mcmc_features_times, f'./data/mcmc_features_times_con_{sigmoid[0]}_X_{sigmoid[2]}.pt')
# torch.save(mcmc_soc_attention, f'./data/mcmc_soc_attention_con_{sigmoid[0]}_X_{sigmoid[2]}.pt')

torch.save(mcmc_features_times, f'./data/mcmc_features_times_con_{sigmoid[0]}_{sigmoid[1]}_X.pt')
torch.save(mcmc_soc_attention, f'./data/mcmc_soc_attention_con_{sigmoid[0]}_{sigmoid[1]}_X.pt')

# torch.save(mcmc_features_times, f'./data/mcmc_features_times_con_{sigmoid[0]}_X_X.pt')
# torch.save(mcmc_soc_attention, f'./data/mcmc_soc_attention_con_{sigmoid[0]}_X_X.pt')